# Climate Change Analysis Notebook

This notebook loads processed climate data and produces summary plots for global temperature anomaly, CO2, and sea level.

In [ ]:
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = pathlib.Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

In [ ]:
temperature = pd.read_parquet(DATA_DIR / 'temperature_monthly.parquet')
co2 = pd.read_parquet(DATA_DIR / 'owid_co2.parquet')
sea_level = pd.read_parquet(DATA_DIR / 'sea_level.parquet')

print('Temperature rows:', len(temperature))
print('CO2 rows:', len(co2))
print('Sea level rows:', len(sea_level))

In [ ]:
temperature.head()

## Temperature anomaly trend

Plot the global monthly temperature anomaly time series.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(temperature['date'], temperature['temperature_anomaly'], color='tab:red', linewidth=1)
ax.set_title('Global Temperature Anomaly (Monthly)')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature anomaly (°C)')
plt.show()

## CO2 emissions and intensity

Plot world-level CO2 values and per-capita CO2 if available.

In [ ]:
world_co2 = co2[co2['country'] == 'World']
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(world_co2['year'], world_co2['co2'], marker='o', label='CO2 emissions (Mt)')
if 'co2_per_capita' in world_co2.columns:
    ax.plot(world_co2['year'], world_co2['co2_per_capita'], marker='o', label='CO2 per capita (t)')
ax.set_title('World CO2 Emissions and Per-Capita CO2')
ax.set_xlabel('Year')
ax.legend()
plt.show()

## Sea level change

Plot the observed sea level dataset.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sea_level['year'], sea_level['sea_level_mm'], color='tab:blue', marker='o')
ax.set_title('Global Sea Level Change')
ax.set_xlabel('Year')
ax.set_ylabel('Sea level change (mm)')
plt.show()

## Summary statistics

This section summarizes dataset coverage and the latest available values from the processed climate data.

In [ ]:
summary = {
    'temperature_rows': len(temperature),
    'co2_rows': len(co2),
    'sea_level_rows': len(sea_level),
    'temperature_range': f"{temperature['date'].min().date()} - {temperature['date'].max().date()}",
    'co2_year_range': f"{int(co2['year'].min())} - {int(co2['year'].max())}",
    'sea_year_range': f"{int(sea_level['year'].min())} - {int(sea_level['year'].max())}",
}
summary

## CO2 and temperature anomaly correlation

Compare world CO2 emissions with annual mean temperature anomaly to explore the relationship between emissions and warming.

In [ ]:
temp_annual = temperature.set_index('date')['temperature_anomaly'].resample('Y').mean().rename('temp_anomaly').to_frame().reset_index()
temp_annual['year'] = temp_annual['date'].dt.year
world_co2 = co2[co2['country'] == 'World'][['year', 'co2']].sort_values('year')
annual = pd.merge(temp_annual, world_co2, on='year', how='inner')
corr = annual['temp_anomaly'].corr(annual['co2'])
fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(annual['co2'], annual['temp_anomaly'], alpha=0.75, color='tab:green')
sns.regplot(x='co2', y='temp_anomaly', data=annual, scatter=False, ax=ax, color='gray')
ax.set_title(f'CO2 emissions vs annual temperature anomaly (corr={corr:.2f})')
ax.set_xlabel('World CO2 emissions (Mt)')
ax.set_ylabel('Annual mean temperature anomaly (°C)')
plt.show()

## Observations

Use these summary figures to support narrative findings, such as the upward trend in temperature and the correlation between CO2 emissions and warming.